In [16]:
# Import necessary libraries
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# Load the dataset
file_path = 'C:/Users/nosao/Desktop/Maxwell-Text Classification/Matrix Response/data/liasion_networking.csv' #Replace with actual file path
df = pd.read_csv(file_path)
df.columns = df.columns.str.strip()  # Clean column names

# Clean missing values
df = df.dropna(subset=['Job description', 'Question 12', 'Question 13', 'Question 14',  'Question 15' ])

# Extract job descriptions and target variables
X = df['Job description'].astype(str)
y_q12 = df['Question 12']
y_q13 = df['Question 13']
y_q14 = df['Question 14']
y_q15 = df['Question 15']

# Split data into training and test sets for each question
X_train, X_test, y_train_q12, y_test_q12 = train_test_split(X, y_q12, test_size=0.2, random_state=42)
X_train_q13, X_test_q13, y_train_q13, y_test_q13 = train_test_split(X, y_q13, test_size=0.2, random_state=42)
X_train_q14, X_test_q14, y_train_q14, y_test_q14 = train_test_split(X, y_q14, test_size=0.2, random_state=42)
X_train_q15, X_test_q15, y_train_q15, y_test_q15 = train_test_split(X, y_q15, test_size=0.2, random_state=42)

# Define Logistic Regression with class_weight='balanced' to handle class imbalance
logreg_pipeline = make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=1000, class_weight='balanced'))

# Hyperparameter tuning grid
param_grid_logreg = {
    'logisticregression__C': [0.01, 0.1, 1, 10],
    'tfidfvectorizer__ngram_range': [(1, 1), (1, 2), (1, 3)],
    'tfidfvectorizer__max_df': [0.85, 0.9, 0.95],
    'tfidfvectorizer__min_df': [1, 5],
    'tfidfvectorizer__use_idf': [True, False],
    'tfidfvectorizer__sublinear_tf': [True, False]
}

# Hyperparameter tuning with StratifiedKFold to ensure balanced class representation in cross-validation
cv = StratifiedKFold(n_splits=5)

# Hyperparameter tuning for Logistic Regression for each question
grid_logreg_q12 = GridSearchCV(logreg_pipeline, param_grid_logreg, cv=cv, scoring='accuracy', n_jobs=-1)
grid_logreg_q12.fit(X_train, y_train_q12)

grid_logreg_q13 = GridSearchCV(logreg_pipeline, param_grid_logreg, cv=cv, scoring='accuracy', n_jobs=-1)
grid_logreg_q13.fit(X_train_q13, y_train_q13)

grid_logreg_q14 = GridSearchCV(logreg_pipeline, param_grid_logreg, cv=cv, scoring='accuracy', n_jobs=-1)
grid_logreg_q14.fit(X_train_q14, y_train_q14)

grid_logreg_q15 = GridSearchCV(logreg_pipeline, param_grid_logreg, cv=cv, scoring='accuracy', n_jobs=-1)
grid_logreg_q15.fit(X_train_q15, y_train_q15)

# Use the best Logistic Regression models for each question
best_logreg_model_q12 = grid_logreg_q12.best_estimator_
best_logreg_model_q13 = grid_logreg_q13.best_estimator_
best_logreg_model_q14 = grid_logreg_q14.best_estimator_
best_logreg_model_q15 = grid_logreg_q15.best_estimator_

# Function to display metrics
def display_metrics(y_true, y_pred, question_num):
    print(f"Metrics for Question {question_num}")
    print(classification_report(y_true, y_pred))
    print(f"Accuracy: {accuracy_score(y_true, y_pred)}\n")

# Predict on the full test set for each question and display metrics
y_pred_q12 = best_logreg_model_q12.predict(X_test)
display_metrics(y_test_q12, y_pred_q12, 12)

y_pred_q13 = best_logreg_model_q13.predict(X_test_q13)
display_metrics(y_test_q13, y_pred_q13, 13)

y_pred_q14 = best_logreg_model_q14.predict(X_test_q14)
display_metrics(y_test_q14, y_pred_q14, 14)

y_pred_q15 = best_logreg_model_q15.predict(X_test_q15)
display_metrics(y_test_q15, y_pred_q15, 15)

# Function to make predictions on a new job description
def predict_for_new_job_description(job_description):
    # Ensure the input is a string
    job_description = [job_description]

    # Predict for each question using the best model
    pred_q12 = best_logreg_model_q12.predict(job_description)[0]
    pred_q13 = best_logreg_model_q13.predict(job_description)[0]
    pred_q14 = best_logreg_model_q14.predict(job_description)[0]
    pred_q15 = best_logreg_model_q15.predict(job_description)[0]

    # Display or return the results
    print("Predictions for the new job description:")
    print(f"Question 12: {pred_q12}")
    print(f"Question 13: {pred_q13}")
    print(f"Question 14: {pred_q14}")
    print(f"Question 15: {pred_q15}")

# Example: Enter a new job description
new_job_description = """
Purpose of the Role: To provide an effective Joinery resource to ensure the University
fabric is efficiently maintained on a day-to-day basis including undertaking Project works. To
ensure the effective interaction of Estate and Facilities services with other services.

Responsible to: Estates Team Leader

Main Duties and Responsibilities:
1. To provide all forms of Joinery duties and tasks in which you are competent within the
University Estate possessing at least five years of trade experience. Working with the team
across various other construction trades.
2. To be responsible for day-to-day breakdown and reactive maintenance.
3. To participate in the Maintenance call-out rota team.
4. To be responsible for working to and delivering cyclical maintenance works ensuring
certain activities are carried out as per the PPM regime.
5. To manage the fire door programme focussing on the Inspection, maintenance (ART
accepted repair techniques), and repair to achieve a compliant campus. This will involve
a good theoretical knowledge of the standards and regulations.
6. To be responsible for the Fire door dashboard, upkeep, and maintenance of the system.
To provide information on defects and repair (analyzing reports), accepted repair
techniques, and input for Projects.
7. To maintain Fire door records on CAFM and in line with the Fire Safety Regulations
(2023). Records must be kept.
8. Identify hazards, defects, and the need for adjustment or repair; to ensure compliance
with agreed codes, law, working practices, and health and safety whilst carrying out your
duties.
9. To provide support and guidance to Contractors engaged in Fire door campus works and
act as a focal point ensuring a fully compliant Fire door install is delivered to the Estate.
10. To manage quantities required to complete each task and manage material stocks and
ordering process.
11. To be responsible for ensuring all tools and equipment are maintained in good working
order and ready for use including power tools within the Estate.
12. To ensure all University fixtures, fittings, furniture, doors, locks, flooring, and other
Joinery items are efficiently maintained, repaired, constructed, or replaced, working
closely with Estates, Facilities Managers, and Estates Team Leader to achieve.
13. To ensure works are delivered in compliance with documented risk assessments and Method
statements, and responsible for the production and review of role-specific risk
assessments.
14. To be responsible for a high standard of conduct always working in a safe and
professional manner reporting any health and safety-related issues to the Estates Team
Leader immediately.
15. To be responsible for working to and delivering all works and repairs in a manner that
ensures VFM and quality finishes are implemented and maintained.
16. To assist the team and organization with general duties over and above your core skills.
Promote, develop and expand the business of our organization generally meeting set
targets.
17. To aid and advise the other members of the University Estates & Facilities staff including
porters and grounds staff as required.
18. To adhere to all organization policies and procedures.
19. To be responsible for continued professional development ensuring the post holder is
conversant and aware of current regulations, legislation, and approved industry
standards to their job role. You will have a basic awareness of Asbestos, CDM,
health and safety regulations.
20. To undertake small projects as reasonably required of the job role and advise all Estates
teams to deliver solutions and cost reductions to all joinery works on campus.

General Duties:
21. To ensure the use of data complies with current regulations, particularly those relating to
GDPR.
22. To comply with all health, safety, and wellbeing policies and procedures at all times and to
take responsibility for promoting and safeguarding the welfare and protection of others.
23. To advocate, promote, and advance equity and social justice within your work.
24. To carry out other duties, commensurate with the grade of the post, as may reasonably be
directed by your line manager after due consultation.
"""

# Call the function with the job description
predict_for_new_job_description(new_job_description)


Metrics for Question 12
              precision    recall  f1-score   support

           C       0.94      1.00      0.97        17
           D       1.00      0.67      0.80         3

    accuracy                           0.95        20
   macro avg       0.97      0.83      0.89        20
weighted avg       0.95      0.95      0.95        20

Accuracy: 0.95

Metrics for Question 13
              precision    recall  f1-score   support

           B       0.65      1.00      0.79        13
           C       0.00      0.00      0.00         5
           D       0.00      0.00      0.00         2

    accuracy                           0.65        20
   macro avg       0.22      0.33      0.26        20
weighted avg       0.42      0.65      0.51        20

Accuracy: 0.65

Metrics for Question 14
              precision    recall  f1-score   support

           A       0.75      0.60      0.67         5
           B       0.75      0.43      0.55         7
           C       0.00  

c:\Users\nosao\Desktop\spacyProj\spacy_venv\Lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\nosao\Desktop\spacyProj\spacy_venv\Lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\nosao\Desktop\spacyProj\spacy_venv\Lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} i